# 🤖 Ollama LLM Backend — Google Colab (GPU T4/A100)

Ce notebook fait tourner Ollama avec un modèle LLM sur le GPU gratuit de Colab.
Il expose l'API via ngrok et se régénère toutes les 24h.

## Modèles supportés (selon GPU)

| GPU | VRAM | Modèle | RAM | Vitesse | Qualité |
|-----|------|--------|-----|---------|---------|
| T4 (free) | 16GB | `qwen2.5:7b` | 5GB | ~30 tok/s | Bonne |
| T4 (free) | 16GB | `qwen2.5:14b` | 10GB | ~15 tok/s | Très bonne |
| T4 (free) | 16GB | `llama3.1:8b` | 5GB | ~35 tok/s | Bonne |
| T4 (free) | 16GB | `mistral:7b` | 5GB | ~35 tok/s | Bonne |
| T4 (free) | 16GB | `deepseek-r1:7b` | 5GB | ~25 tok/s | Raisonnement |
| T4 (free) | 16GB | `deepseek-r1:14b` | 10GB | ~12 tok/s | Raisonnement++ |
| A100 (Pro) | 40GB | `qwen2.5:32b` | 20GB | ~20 tok/s | Excellente |
| A100 (Pro) | 40GB | `llama3.1:70b` | 40GB | ~8 tok/s | Premium |

**Instructions:**
1. Runtime > Change runtime type > GPU (T4 free ou A100 avec Colab Pro)
2. Exécuter toutes les cellules dans l'ordre
3. Choisir le modèle dans la cellule de configuration (Cell 2)
4. Le bot détecte automatiquement le modèle via le webhook

In [ ]:
# Cell 1: Install Ollama + ngrok
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok requests

In [ ]:
# Cell 2: Configuration — Choose your model!

# ─── Model presets ───────────────────────────────────────────
# Pick ONE based on your Colab GPU tier:
#
# T4 (free tier — 16GB VRAM):
#   'qwen2.5:7b'       → Fast, good quality, 5GB RAM, ~30 tok/s
#   'qwen2.5:14b'      → Slower but much better quality, 10GB RAM, ~15 tok/s
#   'llama3.1:8b'      → Meta's model, fast, 5GB RAM, ~35 tok/s
#   'mistral:7b'       → Mistral, fast, 5GB RAM, ~35 tok/s
#   'deepseek-r1:7b'   → Reasoning model, 5GB RAM, ~25 tok/s
#   'deepseek-r1:14b'  → Better reasoning, 10GB RAM, ~12 tok/s
#   'phi4:14b'         → Microsoft, strong reasoning, 10GB RAM, ~15 tok/s
#
# A100 (Colab Pro — 40GB VRAM):
#   'qwen2.5:32b'      → Excellent quality, 20GB RAM, ~20 tok/s
#   'llama3.1:70b'     → Premium quality, 40GB RAM, ~8 tok/s
#   'deepseek-r1:32b'  → Top reasoning, 20GB RAM, ~15 tok/s
#
# ─── Selection ───────────────────────────────────────────────

MODEL_NAME = 'qwen2.5:14b'  # ← Change this to switch models

# Auto-detect GPU type and warn if model too large
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode()
    gpu_name = gpu_info.split(',')[0].strip()
    vram_mb = int(gpu_info.split(',')[1].strip().replace(' MiB', ''))
    print(f'🎮 GPU: {gpu_name} ({vram_mb // 1024}GB VRAM)')
    
    # Rough VRAM requirements (quantized)
    MODEL_VRAM = {
        'qwen2.5:7b': 6, 'qwen2.5:14b': 11, 'qwen2.5:32b': 22,
        'llama3.1:8b': 6, 'llama3.1:70b': 42,
        'mistral:7b': 6, 'mixtral:8x7b': 26,
        'deepseek-r1:7b': 6, 'deepseek-r1:14b': 11, 'deepseek-r1:32b': 22,
        'phi4:14b': 11, 'gemma2:9b': 7, 'gemma2:27b': 18,
    }
    needed_gb = MODEL_VRAM.get(MODEL_NAME, 10)
    available_gb = vram_mb // 1024
    if needed_gb > available_gb:
        print(f'⚠️  WARNING: {MODEL_NAME} needs ~{needed_gb}GB VRAM but GPU has {available_gb}GB!')
        print(f'   Consider using a smaller model or upgrading to Colab Pro (A100).')
    else:
        print(f'✅ {MODEL_NAME} fits in {available_gb}GB VRAM (needs ~{needed_gb}GB)')
except Exception as e:
    print(f'⚠️ Could not detect GPU: {e}')

# ─── Other settings ──────────────────────────────────────────
NGROK_AUTHTOKEN = ''  # <-- PASTE YOUR NGROK AUTHTOKEN HERE
REGENERATE_HOURS = 24  # Session regeneration interval

# Webhook URL to notify the bot of URL changes
# The bot will auto-switch to the new URL + model
BOT_WEBHOOK_URL = ''  # e.g. https://your-vps.com/webhook/colab-url

# Keep model warm in VRAM between requests (avoids reload delay)
KEEP_ALIVE = '30m'  # Ollama keep_alive: '5m', '30m', '1h', or '-1' (forever)

print(f'\n📋 Configuration:')
print(f'   Model: {MODEL_NAME}')
print(f'   Regeneration: every {REGENERATE_HOURS}h')
print(f'   Keep-alive: {KEEP_ALIVE}')

In [ ]:
# Cell 3: Start Ollama server + pull model
import subprocess, os, time, signal

# Kill any existing Ollama
os.system('pkill ollama')
time.sleep(2)

# Start Ollama server with GPU optimization
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
env['OLLAMA_MAX_LOADED_MODELS'] = '1'
env['OLLAMA_NUM_PARALLEL'] = '2'
env['OLLAMA_KEEP_ALIVE'] = KEEP_ALIVE
# Force GPU usage
env['OLLAMA_GPU_OVERHEAD'] = '0'

ollama_proc = subprocess.Popen(['ollama', 'serve'],
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                env=env)
time.sleep(3)
print('✅ Ollama server started (GPU mode)')

# Pull model
print(f'⏳ Pulling model {MODEL_NAME}...')
print(f'   This may take 2-10 minutes depending on model size...')
pull_result = os.system(f'ollama pull {MODEL_NAME}')
if pull_result == 0:
    print(f'✅ Model {MODEL_NAME} ready')
    
    # Verify model is loaded on GPU
    try:
        result = subprocess.check_output(['ollama', 'show', MODEL_NAME], stderr=subprocess.DEVNULL).decode()
        if 'GPU' in result or 'gpu' in result:
            print(f'   🎮 Model loaded on GPU')
        print(f'   📊 Model info:')
        for line in result.split('\n')[:10]:
            print(f'      {line}')
    except:
        pass
else:
    print(f'❌ Failed to pull {MODEL_NAME}')
    raise RuntimeError(f'Pull failed for {MODEL_NAME}')

In [ ]:
# Cell 4: Start ngrok tunnel + notify bot
from pyngrok import ngrok, conf
import requests, json

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

# Kill existing tunnels
ngrok.kill()
time.sleep(2)

# Start tunnel on port 11434 (Ollama default)
tunnel = ngrok.connect(11434, 'http')
OLLAMA_URL = tunnel.public_url
print(f'🌐 Ollama public URL: {OLLAMA_URL}')
print(f'   → Bot will auto-detect via webhook or URL file')

# Notify bot webhook if configured — sends URL + model name
# The bot uses the model name to configure its OpenAI client
if BOT_WEBHOOK_URL:
    try:
        resp = requests.post(BOT_WEBHOOK_URL, json={
            'url': OLLAMA_URL,
            'model': MODEL_NAME,
            'gpu': gpu_name if 'gpu_name' in dir() else 'unknown',
            'keep_alive': KEEP_ALIVE,
        }, timeout=10)
        print(f'📡 Bot notified: {resp.status_code} — model={MODEL_NAME}')
    except Exception as e:
        print(f'⚠️ Webhook failed: {e}')
        print(f'   Bot will poll URL file instead')

In [ ]:
# Cell 5: Keep-alive + auto-regeneration loop
# This cell runs until Colab session expires, then you re-run the notebook

import time, requests, json, subprocess
from datetime import datetime, timedelta

start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=REGENERATE_HOURS)

print(f'🕐 Session started at {start_time.strftime("%H:%M:%S")}')
print(f'🔄 Will regenerate at {regenerate_at.strftime("%H:%M:%S")} ({REGENERATE_HOURS}h)')
print(f'📊 Monitoring loop started (checks every 60s)...')
print(f'   Press Ctrl+C to stop manually')

health_ok_count = 0
health_fail_count = 0

try:
    while True:
        now = datetime.now()
        elapsed = now - start_time
        remaining = regenerate_at - now
        
        # Health check Ollama
        try:
            resp = requests.get(f'{OLLAMA_URL}/api/tags', timeout=10)
            if resp.status_code == 200:
                health_ok_count += 1
                status = '✅ healthy'
            else:
                health_fail_count += 1
                status = f'⚠️ status {resp.status_code}'
        except Exception as e:
            health_fail_count += 1
            status = f'❌ {str(e)[:50]}'
        
        # Print status every 5 minutes
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            print(f'[{now.strftime("%H:%M:%S"}] uptime={int(elapsed.total_seconds()/60)}min '
                  f'remaining={int(remaining.total_seconds()/60)}min '
                  f'ok={health_ok_count} fail={health_fail_count} {status}')
        
        # Check if it's time to regenerate
        if now >= regenerate_at:
            print(f'🔄 Regeneration triggered at {now.strftime("%H:%M:%S")}')
            print('   Restarting ngrok tunnel...')
            
            # Kill old tunnel
            ngrok.kill()
            time.sleep(3)
            
            # Start new tunnel
            new_tunnel = ngrok.connect(11434, 'http')
            OLLAMA_URL = new_tunnel.public_url
            print(f'   New URL: {OLLAMA_URL}')
            
            # Notify bot
            if BOT_WEBHOOK_URL:
                try:
                    requests.post(BOT_WEBHOOK_URL, json={'url': OLLAMA_URL, 'model': MODEL_NAME})
                    print('   📡 Bot notified of new URL')
                except Exception as e:
                    print(f'   ⚠️ Webhook failed: {e}')
            
            # Reset timer
            regenerate_at = now + timedelta(hours=REGENERATE_HOURS)
            print(f'   Next regeneration at {regenerate_at.strftime("%H:%M:%S")}')
        
        time.sleep(60)
        
except KeyboardInterrupt:
    print('\n⏹️ Stopped by user')
except Exception as e:
    print(f'\n❌ Error: {e}')
finally:
    print(f'Session stats: ok={health_ok_count} fail={health_fail_count}')

In [ ]:
# Cell 6 (optional): Test the LLM
import requests, json

test_url = f'{OLLAMA_URL}/v1/chat/completions'
payload = {
    'model': MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Bonjour, réponds en une phrase.'}],
    'max_tokens': 50,
    'stream': False
}

resp = requests.post(test_url, json=payload, timeout=60)
print(f'Status: {resp.status_code}')
if resp.status_code == 200:
    data = resp.json()
    print(f'Response: {data["choices"][0]["message"]["content"]}')
else:
    print(f'Error: {resp.text[:200]}')